In [ ]:
!pip install deepxde

In [ ]:
import numpy as np
import tensorflow.compat.v1 as tf
import deepxde as dde
import matplotlib.pyplot as plt

tf.disable_v2_behavior()

# Load an OpenFOAM CSV export
def preprocess_data(file_path, velocity_initial):
    data_csv = np.loadtxt(file_path, delimiter=",", skiprows=1)
    coords = data_csv[:, 1:4]  # Coordinates (x, y, z)
    velocities = data_csv[:, 4:6]  # OpenFOAM velocities (U:0, U:1)
    velocity_column = np.full((coords.shape[0], 1), velocity_initial)  # Inlet-velocity column
    return coords, velocities, velocity_column

# Datasets and their inlet velocities
file_path_train1 = "/content/2_datos_Caso1_AMIMF_300.csv"
velocity_initial_train1 = 1

file_path_train2 = "/content/2_caso2_AMIMF_899.csv"
velocity_initial_train2 = 10

file_path_test = "/content/2_datos_T_AMIMF_899.csv"
velocity_initial_test = 5

# Preprocess training and test data
coords_train1, outputs_train1, velocity_train1 = preprocess_data(file_path_train1, velocity_initial_train1)
coords_train2, outputs_train2, velocity_train2 = preprocess_data(file_path_train2, velocity_initial_train2)
coords_test, outputs_test, velocity_test = preprocess_data(file_path_test, velocity_initial_test)

# Merge training data
coords_train = np.concatenate([coords_train1, coords_train2], axis=0)
outputs_train = np.concatenate([outputs_train1, outputs_train2], axis=0)
velocity_train = np.concatenate([velocity_train1, velocity_train2], axis=0)

# Sanity checks
for coords, outputs in [(coords_train, outputs_train), (coords_test, outputs_test)]:
    assert not np.any(np.isnan(coords)), "Coordinates contain NaN values"
    assert not np.any(np.isinf(coords)), "Coordinates contain Inf values"
    assert not np.any(np.isnan(outputs)), "Velocities contain NaN values"
    assert not np.any(np.isinf(outputs)), "Velocities contain Inf values"


# Computational domain (cuboid)
cuboid = dde.geometry.Cuboid([-20, -20, -0.5], [30, 20, 0.5])

# Fully connected network
layer_size = [3] + [65] * 4 + [2]  # 4 hidden layers
net = dde.maps.FNN(layer_size, "tanh", "Glorot normal")

# Boundary conditions
def boundary_inlet(x, on_boundary):
    return on_boundary and np.isclose(x[0], -20)  # Inlet at x = -20

def boundary_outlet(x, on_boundary):
    return on_boundary and np.isclose(x[0], 30)  # Outlet at x = 30

def inlet_velocity(x):
    return [velocity_initial_train1, 0.0]
bc_inlet = dde.DirichletBC(cuboid, inlet_velocity, boundary_inlet)
bc_outlet = dde.NeumannBC(cuboid, lambda x: 0, boundary_outlet)

# Steady incompressible Navier-Stokes residuals (x-y plane)
def navier_stokes(x, y):
    rho = 1000
    mu = 0.001

    u, v = y[:, 0:1], y[:, 1:2]

    u_x, u_y = dde.grad.jacobian(y, x, i=0, j=0), dde.grad.jacobian(y, x, i=0, j=1)
    v_x, v_y = dde.grad.jacobian(y, x, i=1, j=0), dde.grad.jacobian(y, x, i=1, j=1)

    laplacian_u = dde.grad.hessian(y, x, component=0, i=0, j=0) + dde.grad.hessian(y, x, component=0, i=1, j=1)
    laplacian_v = dde.grad.hessian(y, x, component=1, i=0, j=0) + dde.grad.hessian(y, x, component=1, i=1, j=1)

    continuity = u_x + v_y

    momentum_u = rho * (u * u_x + v * u_y) - mu * laplacian_u
    momentum_v = rho * (u * v_x + v * v_y) - mu * laplacian_v

    return [continuity, momentum_u, momentum_v]

data = dde.data.PDE(
    geometry=cuboid,
    pde=navier_stokes,
    bcs=[bc_inlet, bc_outlet],
    num_domain=5000,
    num_boundary=2000,
    anchors=coords_train,
)


model = dde.Model(data, net)

model.compile("adam", lr=0.001, loss="mse")

# Train
losshistory, train_state = model.train(epochs=1000)

# Loss history
plt.figure(figsize=(10, 6))
plt.plot(losshistory.steps, losshistory.loss_train, label="Training loss")
plt.plot(losshistory.steps, losshistory.loss_test, label="Test loss")
plt.xlabel("Iterations")
plt.ylabel("Loss")
plt.title("Training loss history")
plt.legend()
plt.grid(True)
plt.show()


# Evaluate on the held-out inlet velocity
y_pred_test = model.predict(coords_test)
print("Predicted velocities at sample test points:")
print(y_pred_test[:10])

# Relative L2 error against OpenFOAM
relative_error = np.linalg.norm(outputs_test - y_pred_test) / np.linalg.norm(outputs_test)
print(f"Relative L2 error (test): {relative_error:.4f}")

# Plot test-case results
x_test, y_test = coords_test[:, 0], coords_test[:, 1]
u_pred_test, v_pred_test = y_pred_test[:, 0], y_pred_test[:, 1]
u_real_test, v_real_test = outputs_test[:, 0], outputs_test[:, 1]

# Predicted velocity field
plt.figure(figsize=(10, 6))
plt.quiver(x_test, y_test, u_pred_test, v_pred_test, scale=50, color="blue", alpha=0.6)
plt.title("Simulated velocity field for train case (5 m/s), (U:0, U:1)")
plt.xlabel("x")
plt.ylabel("y")
plt.axis("equal")
plt.show()

# OpenFOAM velocity field
plt.figure(figsize=(10, 6))
plt.quiver(x_test, y_test, u_real_test, v_real_test, scale=50, color="red", alpha=0.6)
plt.title("Real velocity field for train case (5 m/s), (U:0, U:1)")
plt.xlabel("x")
plt.ylabel("y")
plt.axis("equal")
plt.show()
